# Generate final count table for duplicate sgRNA logic  
a part of the logic from /media/scratch/fy2306/projects/base_editing/script/archive/grna_plot_lfc_seq_wt.ipynb  
per nucleotide

In [1]:
import pandas as pd
import numpy as np

In [2]:
seq_name_list = [
	"MYC_GFP", "MYC_SNP1", "MYC_SNP2", "MYC_SNP3", "MYC_STOP"
]

myc_consequence_list = [
	"upstream_flanking",
	"downstream_flanking",
	"5_prime_UTR",
	"3_prime_UTR",
	"intron",
	"coding",
	"intron_splice"
]

In [3]:
def duplicate_count_table(df_count_path, df_lib_path, df_count_out_path, df_ctrl_out_path, window_side_size, edited_pos_chr_col="indel_base_pos_chr"):
	df_count = pd.read_csv(df_count_path, sep="\t")
	df_lib = pd.read_csv(df_lib_path, sep="\t", usecols=['id', 'seq_name', edited_pos_chr_col, 'grna_target_seq_rc_strand', 'indel_consequences', 'indel_repeats'])

	df = df_lib[(df_lib["seq_name"].isin(seq_name_list)) & (df_lib["indel_consequences"].isin(myc_consequence_list)) & (df_lib["indel_repeats"] == False)]

	# Sort by genomic position
	df = df.sort_values(by=edited_pos_chr_col).reset_index(drop=True)
	# edited_pos_chr_col_cut: from base pos to between-base pos
	edited_pos_chr_col_cut = edited_pos_chr_col + "_cutting_site"
	df[edited_pos_chr_col_cut] = np.where(
		df['grna_target_seq_rc_strand'] == '-',
		df[edited_pos_chr_col] + 0.5,
		df[edited_pos_chr_col] - 0.5
	)

	# Apply rolling mask manually
	# inclusive 2*window_side_size window (e.g. if window_side_size = 3 -> 6 cutting sites with nuleotide-of-interest at the center)
	# inclusive start and end
	df['gRNA_cover_Start'] = np.where(
		df['grna_target_seq_rc_strand'] == '-',
		df[edited_pos_chr_col] - 2,
		df[edited_pos_chr_col] - 17
	)
	df['gRNA_cover_End'] = np.where(
		df['grna_target_seq_rc_strand'] == '-',
		df[edited_pos_chr_col] + 17,
		df[edited_pos_chr_col] + 2
	)
	min_val = int(df[['gRNA_cover_Start', 'gRNA_cover_End']].min().min())
	max_val = int(df[['gRNA_cover_Start', 'gRNA_cover_End']].max().max())

	nucleotide_df = pd.DataFrame({
		'nucleotide_pos': range(min_val, max_val + 1)
	})
	nucleotide_df["win_start"] = nucleotide_df["nucleotide_pos"] - window_side_size
	nucleotide_df["win_end"] = nucleotide_df["nucleotide_pos"] + window_side_size

	# mapping
	sgRNA_to_masked_pos = {}

	for idx, row in nucleotide_df.iterrows():
		pos = row["nucleotide_pos"]
		start = row["win_start"]
		end = row["win_end"]
		
		# Find sgRNAs (ids) within this window
		mask = (df[edited_pos_chr_col_cut] > start) & (df[edited_pos_chr_col_cut] < end)
		contributing_ids = df.loc[mask, 'id'].values
		
		for sg in contributing_ids:
			if sg not in sgRNA_to_masked_pos:
				sgRNA_to_masked_pos[sg] = []
			if pos not in sgRNA_to_masked_pos[sg]:
				sgRNA_to_masked_pos[sg].append(pos)

	expanded_rows = []

	# Loop over df_count rows
	for _, row in df_count.iterrows():
		sg = row['sgRNA']
		masked_positions = sgRNA_to_masked_pos.get(sg)
		
		# Skip if no positions to expand to
		if not masked_positions:
			continue
		
		for pos in masked_positions:
			# Create a copy of the row as a dictionary
			new_row = row.to_dict()
			
			# Modify 'sgRNA' and 'Gene' columns
			new_row['sgRNA'] = f"{row['sgRNA']}_{pos}"
			# new_row['Gene'] = f"{row['Gene']}_{pos}" split MYC_GFP, MYC_SNP and MYC_STOP
			new_row['Gene'] = f"MYC_GFP_{pos}"
			
			expanded_rows.append(new_row)


	df_expanded = pd.DataFrame(expanded_rows)

	# add back the rest of the sgRNAs
	df_rest = df_count[df_count["Gene"].isin(["NT", "Random", "AAVS1"])]
	n_copies = 2*window_side_size
	# n_copies = 1

	df_duplicated = pd.DataFrame(df_rest.values.repeat(n_copies, axis=0), columns=df_rest.columns)

	suffixes = np.tile(np.arange(1, n_copies + 1), len(df_rest))
	suffix_str = suffixes.astype(str)

	df_duplicated = df_duplicated.reset_index(drop=True)

	df_duplicated['sgRNA'] = df_duplicated['sgRNA'].astype(str) + '-' + suffix_str
	df_duplicated['Gene'] = df_duplicated['Gene'].astype(str) + '-' + suffix_str

	df_duplicated["sgRNA"].to_csv(df_ctrl_out_path, index=False, header=False)

	df_expanded = pd.concat([df_expanded, df_duplicated], ignore_index=True)
	df_expanded.to_csv(df_count_out_path, sep="\t", index=False)

In [4]:
window_side_size = 2
duplicate_count_table(
	df_count_path="/media/scratch/fy2306/projects/base_editing/data/20250416/mageck/test-1-standard/count/MYC_U1.count_normalized.txt", 
	df_lib_path="/media/scratch/fy2306/projects/base_editing/data/grna_type/MYC-lib-for-mageck.type.myc1&2.organized.txt", 
	df_count_out_path=f"/media/scratch/fy2306/projects/base_editing/data/20250416/mageck/test-1-standard-nucleotide/count/MYC_U1.count_normalized.{str(int(2*window_side_size))}duplicated.no_repeats.txt",
	df_ctrl_out_path=f"/media/scratch/fy2306/projects/base_editing/data/20250416/mageck/test-1-standard-nucleotide/count/control_sgrna.{str(int(2*window_side_size))}duplicated.txt",
	window_side_size=window_side_size,
	edited_pos_chr_col="indel_base_pos_chr"
	)